In [3]:
%pip -q install duckdb huggingface_hub

In [4]:
import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

Paste your Hugging Face READ token (hf_...): ··········


In [5]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("Connected successfully!")

Connected successfully!


Create the feature table

In [7]:
# Build features from the last 60 days

features = con.sql(f"""
WITH bounds AS (
    SELECT MAX(report_date) AS end_d
    FROM {TABLES['fact_daily']}
),

windowed AS (
    SELECT
        f.client_hash_id,
        f.content_hash_id,

        SUM(
            CASE
                WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                THEN f.gsc_impressions
                ELSE 0
            END
        ) AS imp_last30,

        SUM(
            CASE
                WHEN f.report_date <= b.end_d - INTERVAL 30 DAY
                THEN f.gsc_impressions
                ELSE 0
            END
        ) AS imp_prev30,

        SUM(
            CASE
                WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                THEN f.gsc_clicks
                ELSE 0
            END
        ) AS clk_last30,

        AVG(
            CASE
                WHEN f.report_date > b.end_d - INTERVAL 30 DAY
                THEN f.gsc_avg_position
            END
        ) AS pos_last30

    FROM {TABLES['fact_daily']} f,
         bounds b

    WHERE f.report_date > b.end_d - INTERVAL 60 DAY

    GROUP BY
        f.client_hash_id,
        f.content_hash_id

    HAVING imp_prev30 >= 100
)
SELECT *
FROM windowed
""").df()

print(f"Total pages: {len(features):,}")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total pages: 111,247


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30
0,client_e547b89c05043229,content_6b80dfab2e0ffa2e,1110.0,955.0,12.0,7.543789
1,client_e547b89c05043229,content_d7bb60ec9a42c11a,3735.0,3338.0,33.0,5.446636
2,client_e547b89c05043229,content_401dcc5cd616e3dd,181.0,130.0,0.0,6.874167
3,client_e547b89c05043229,content_18d95bd7890430ed,151.0,340.0,0.0,33.665367
4,client_e547b89c05043229,content_56f46c55f0348ab4,392.0,531.0,3.0,12.995100


In [8]:
# Build query-level features

qsignals = con.sql(f"""
SELECT
    content_hash_id,

    ANY_VALUE(content_visible_query_count) AS visible_queries,

    ANY_VALUE(rare_impressions_share) AS rare_share,

    ANY_VALUE(anonymized_impressions_share) AS anon_share,

    MAX(impressions_90d) AS top_query_impressions,

    SUM(impressions_90d) AS kept_impressions

FROM {TABLES['fact_query_90d']}

GROUP BY content_hash_id
""").df()

# Calculate the share of impressions from the top query
qsignals["top_query_share"] = (
    qsignals["top_query_impressions"] /
    qsignals["kept_impressions"]
)

# Merge with the feature table
data = features.merge(
    qsignals,
    on="content_hash_id",
    how="left"
)

print(f"Final dataset: {len(data):,} rows")
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Final dataset: 111,247 rows


,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_e547b89c05043229,content_6b80dfab2e0ffa2e,1110.0,955.0,12.0,7.543789,1.0,0.022750,0.957216,59.0,59.0,1.000000
1,client_e547b89c05043229,content_d7bb60ec9a42c11a,3735.0,3338.0,33.0,5.446636,14.0,0.017946,0.932994,84.0,462.0,0.181818
2,client_e547b89c05043229,content_401dcc5cd616e3dd,181.0,130.0,0.0,6.874167,3.0,0.162037,0.552469,153.0,185.0,0.827027
3,client_e547b89c05043229,content_18d95bd7890430ed,151.0,340.0,0.0,33.665367,2.0,0.108932,0.820261,52.0,65.0,0.800000
4,client_e547b89c05043229,content_56f46c55f0348ab4,392.0,531.0,3.0,12.995100,5.0,0.163052,0.788332,14.0,65.0,0.215385


Create the Label and Train the Random Forest

In [9]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# -----------------------------
# Create the target label
# -----------------------------

data["is_declining"] = (
    data["imp_last30"] < 0.8 * data["imp_prev30"]
).astype(int)

# -----------------------------
# Select features
# -----------------------------

feature_cols = [
    "imp_prev30",
    "visible_queries",
    "rare_share",
    "anon_share",
    "top_query_share"
]

model_data = data.dropna(subset=feature_cols)

X = model_data[feature_cols]
y = model_data["is_declining"]

# -----------------------------
# Train/Test Split
# -----------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# -----------------------------
# Train Random Forest
# -----------------------------

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# -----------------------------
# Predictions
# -----------------------------

predictions = model.predict(X_test)

# -----------------------------
# Results
# -----------------------------

accuracy = accuracy_score(y_test, predictions)

print(f"Accuracy: {accuracy:.4f}")

print("\nClassification Report")
print(classification_report(y_test, predictions))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, predictions))

Accuracy: 0.6545

Classification Report
              precision    recall  f1-score   support

           0       0.55      0.34      0.42      9389
           1       0.69      0.84      0.75     16162

    accuracy                           0.65     25551
   macro avg       0.62      0.59      0.59     25551
weighted avg       0.64      0.65      0.63     25551


Confusion Matrix
[[ 3164  6225]
 [ 2604 13558]]


# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mirabdulbaqi/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question
## Research Question

Can machine learning identify content pages that should be prioritized for review by using search performance signals such as click-through rate (CTR), average search position, impressions, clicks, and engagement?

## Decision Supported

This project supports content managers and SEO teams in deciding which pages should be reviewed first. Instead of manually checking thousands of pages, the model ranks content based on observed search performance and recommends pages that may benefit from optimization. The recommendations are intended as decision support rather than final decisions.


In [1]:
print("Research Question")

print("-" * 50)

print("Can machine learning identify content pages that should be prioritized for review using search performance signals?")

print("\nDecision Supported")

print("- Rank pages for review")
print("- Support SEO/content decisions")
print("- Decision-support only")

Research Question
--------------------------------------------------
Can machine learning identify content pages that should be prioritized for review using search performance signals?

Decision Supported
- Rank pages for review
- Support SEO/content decisions
- Decision-support only


## 2. Data

This project uses the FlyRank ML Internship warehouse hosted on Hugging Face. The analysis is based on the public-safe, pseudonymized Search Intelligence dataset.

The primary tables used are:

* **fact_content_daily_performance** – daily search performance metrics for content.
* **fact_content_query_90d** – query-level search signals.
* **dim_content** – content information.
* **dim_clients** – client metadata used for filtering and validation.

For feature engineering and model development, the analysis focuses on a mid-panel month (March 2026). This avoids using the final month as a development window and follows the internship guidance to reduce the risk of data leakage.

The project excludes client names, domains, URLs, private search queries, credentials, and any future information that would not be available at the prediction time. Only observed historical search performance signals are used.


In [6]:
# Show basic information about the data used
print("Dataset: FlyRank ML Internship Warehouse")
print("Analysis Month: March 2026")

print("\nTables used:")

for table in TABLES:
    print("-", table)

Dataset: FlyRank ML Internship Warehouse
Analysis Month: March 2026

Tables used:
- dim_clients
- dim_content
- fact_daily
- fact_daily_sample
- fact_query_90d


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
